# Strategic Blocking: Colab transfer v2
Run in a fresh GPU runtime. Upload `colab-transfer-v2.zip` to `MyDrive/strategic-blocking-transfer/` first.
This is an inference-only evaluation of the same base / before / after policies and eight maps.
The v1 run is retained as an invalid policy comparison caused by near-universal format rejection.
V2 ports the captured RAGEN text interface; FP16 Transformers sampling remains a separate backend.
These maps have already been inspected in the v1 format investigation. Do not call v2 newly blind.


In [ ]:
import importlib.metadata, subprocess, sys
try:
    previous_torchao = importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError:
    previous_torchao = None
setup_record = {'previous_torchao': previous_torchao, 'reason': 'Optional torchao is unused by this non-quantized FP16 protocol; v1 encountered a PEFT import conflict.'}
if previous_torchao is not None:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers==4.56.1', 'peft==0.21.2', 'accelerate==1.15.0', 'safetensors==0.8.0', 'pyyaml==6.0.3'], check=True)
print(setup_record)


If model libraries were already imported, restart the runtime. Keep Colab's CUDA Torch. Run the setup cell in a fresh runtime before continuing.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile, tempfile, json, hashlib
archive = Path('/content/drive/MyDrive/strategic-blocking-transfer/colab-transfer-v2.zip')
if not archive.is_file():
    raise FileNotFoundError(f'Upload the v2 handoff ZIP first: {archive}')
bundle = Path(tempfile.mkdtemp(prefix='strategic-blocking-v2-', dir='/content'))
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        if not (bundle / item.filename).resolve().is_relative_to(bundle.resolve()):
            raise ValueError('Unsafe ZIP path')
    z.extractall(bundle)
manifest = json.loads((bundle / 'bundle-manifest.json').read_text())
for name, digest in manifest.items():
    target = (bundle / name).resolve()
    if not target.is_relative_to(bundle.resolve()):
        raise ValueError('Unsafe manifest path')
    assert hashlib.sha256(target.read_bytes()).hexdigest() == digest, name
output_parent = Path('/content/drive/MyDrive/strategic-blocking-transfer/results')
output_parent.mkdir(parents=True, exist_ok=True)
setup_file = output_parent / ('setup-v2-' + bundle.name + '.json')
setup_file.write_text(json.dumps(setup_record, indent=2) + '\n')
runner = bundle / 'scripts/evaluate_colab_transfer_v2.py'
def run(*arguments):
    process = subprocess.Popen([sys.executable, '-u', str(runner), '--bundle', str(bundle),
                                '--output-parent', str(output_parent), *arguments],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    destination = None
    for line in process.stdout:
        print(line, end='', flush=True)
        if line.startswith('Output: '):
            destination = Path(line[len('Output: '):].strip())
    if process.wait() != 0:
        raise RuntimeError('Evaluation failed. Preserve the output and inspect FAILED.json; do not continue.')
    if destination is not None:
        (destination / 'setup.json').write_text(json.dumps(setup_record, indent=2) + '\n')
    return destination
run('--verify-only')


## Required format smoke test
45 episodes: 5 episodes on each of 3 old development maps, for each policy.
Each policy must have at least 15 turns and at most 5% format errors. There is no win-rate gate.
The full run verifies the raw smoke evidence and identical code, tokenizer, adapters and runtime.
Do not bypass or loosen a failed gate; share that run for diagnosis.


In [ ]:
subprocess.run([sys.executable, str(bundle / 'scripts/validate_frozen_suite.py')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'test_colab_v2', '-v'], cwd=bundle, check=True)
smoke_output = run('--smoke')
smoke_report = smoke_output / 'quality-gate.json'
print(smoke_report.read_text())


## Full evaluation after smoke passes
Run once: 8 maps x 3 policies x 50 episodes = 1,200 episodes, separate from smoke.
A continuing format-error guard aborts a broken run. Budget exhaustion is reported as unresolved.
Do not merge this result with RAGEN or v1. Share the entire output folder, including raw trajectories.


In [ ]:
full_output = run('--smoke-report', str(smoke_report))
print((full_output / 'comparison.md').read_text())
print('Send this folder:', full_output)
